In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

(a)

In [ ]:
signal_path = "data/signal_Bs2MuMu.txt"
background_path = "data/background_combinatorial.txt"
features = ["PT1", "PT2", "P1", "P2", "TotalPT", "VertexChisq", "Isolation", "MASS"]

signal = pd.read_csv(signal_path, sep=r"\s+", header=None, names=features)
background = pd.read_csv(background_path, sep=r"\s+", header=None, names=features)
signal = signal.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)
background = background.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)

os.makedirs("plots", exist_ok=True)

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, feat in zip(axes.flatten(), features):
    all_vals = np.concatenate([signal[feat].values, background[feat].values])
    bins = np.linspace(all_vals.min(), all_vals.max(), 50)
    ax.hist(background[feat], bins=bins, alpha=0.6, label="Background", density=True)
    ax.hist(signal[feat], bins=bins, alpha=0.6, label="Signal", density=True)
    ax.set_title(feat); ax.set_xlabel(feat); ax.set_ylabel("Density"); ax.legend()

plt.tight_layout()
plt.savefig("plots/all_features_overview.png", dpi=300, bbox_inches="tight")
plt.show()

All features can be used except the mass as we will use the mass distribution to fit a signal peak over the background. Using mss in the classifier would therefore bias the distribution 

(b)

In [ ]:
def fisher_score(sig_col, bkg_col):
    sig_col = np.asarray(sig_col, dtype=float)
    bkg_col = np.asarray(bkg_col, dtype=float)
    all_vals = np.concatenate([sig_col, bkg_col])
    mu_sig, mu_bkg = sig_col.mean(), bkg_col.mean()
    mu_all = all_vals.mean()
    sigma2 = all_vals.var()
    if sigma2 == 0:
        return 0.0
    return (len(sig_col) * (mu_sig - mu_all) ** 2 + len(bkg_col) * (mu_bkg - mu_all) ** 2) / sigma2

scores = [{"Feature": f, "FisherScore": fisher_score(signal[f], background[f])} for f in features]
fisher_df = pd.DataFrame(scores).sort_values("FisherScore", ascending=False).reset_index(drop=True)
fisher_df["Rank"] = np.arange(1, len(fisher_df) + 1)
fisher_df = fisher_df[["Rank", "Feature", "FisherScore"]]
display(fisher_df)
fisher_df.to_csv("fisher_scores.csv", index=False)